# Drum Practice — servidor Demucs para Xiaomi (Colab)

Ejecutá **Entorno de ejecución → Ejecutar todas**. Permití la ejecución. Colab debe permanecer **conectado** y esta pestaña abierta durante la separación. El enlace HTTPS será temporal y cambiará al reiniciar. No publiques el enlace ni la clave. **Puede tardar bastante en CPU; la GPU gratuita no está garantizada.**

Este cuaderno crea un túnel temporal de Cloudflare, sin instalar nada en el Xiaomi. Las canciones que subas atraviesan el túnel y se procesan en la sesión de Colab. No uses grabaciones confidenciales.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install 'demucs==4.0.1' 'fastapi>=0.115,<1' 'uvicorn>=0.30,<1' 'python-multipart>=0.0.9'
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared


In [ ]:
import os, secrets, subprocess, time, re, pathlib, urllib.request, shutil
from google.colab import files
# Subí SOLO app.py desde la carpeta server del ZIP v0.7.
uploaded = files.upload()
assert 'app.py' in uploaded, 'Debés subir app.py del ZIP v0.7'
pathlib.Path('/content/app.py').write_bytes(uploaded['app.py'])
os.environ['DRUM_API_KEY'] = secrets.token_urlsafe(24)
print('CLAVE PARA LA APP:', os.environ['DRUM_API_KEY'])
server_log = open('/content/drum_server.log','w')
server = subprocess.Popen(['python','-m','uvicorn','app:app','--host','127.0.0.1','--port','8000'], cwd='/content', env=os.environ.copy(), stdout=server_log,stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=2)
        break
    except Exception:time.sleep(1)
else:raise RuntimeError('Servidor no inicia: '+pathlib.Path('/content/drum_server.log').read_text()[-2000:])
print('Servidor local activo')


In [ ]:
tunnel_log = open('/content/drum_tunnel.log','w')
tunnel = subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(80):
    text = pathlib.Path('/content/drum_tunnel.log').read_text()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',text)
    if match:
        url=match.group(0);break
    if tunnel.poll() is not None:break
    time.sleep(1)
if not url:raise RuntimeError('No se obtuvo el enlace HTTPS: '+text[-2500:])
print('URL DEL SERVIDOR PARA ANDROID:',url)
print('CLAVE DEL SERVIDOR:',os.environ['DRUM_API_KEY'])
print('Dejá Colab conectado. Al desconectarse, el servidor deja de funcionar.')


## En el Xiaomi

1. Instalá una APK creada a partir del proyecto Android incluido en el ZIP (todavía no hay APK compilado).
2. En «Separación IA por Internet», pegá **URL DEL SERVIDOR PARA ANDROID** y **CLAVE DEL SERVIDOR** mostradas por Colab.
3. Elegí una canción MP3 de hasta 35 MB, tocá **Separar batería con IA (Internet)** y esperá el resultado.
4. Las pistas descargadas se podrán escuchar offline desde la app.

**Importante:** Colab gratuito no es un servidor 24/7. Puede desconectarse; si lo hace, reiniciá las celdas y actualizá URL y clave. Las canciones son procesadas en una nube de terceros; el túnel es público pero el endpoint requiere la clave.
